---
# Fall 2023 MATH 760 Case Study: Network Flow Optimization

In [12]:
using GLPK

---

## Introduction

We will look at a variety of different types of **network flow problems** to analyze public transit in Vancouver. The network flow problems we study here are linear programs which can be very efficiently solved using the simplex method.

Let $G = (V, E)$ be a **directed graph** with node set $V$ and edge set $E \subseteq V \times V$. We will let $x_{ij} \ge 0$ represent the amount of **flow** (e.g., the number of people per hour) from node $i$ to node $j$ along the edge $(i,j)$. We can write the total amount of flow leaving node $i$ as $\sum_{j:(i,j) \in E} x_{ij}$, and the amount of flow arriving at node $i$ as $\sum_{j:(j,i) \in E} x_{ji}$. Let

$$
\delta_x(i) := \sum_{j:(j,i) \in E} x_{ji} - \sum_{j:(i,j) \in E} x_{ij}
$$

represent the **net flow** at node $i$.

A basic constraint in many network flow problems is that the amount of flow going into a node must be equal to the amount of flow leaving that node. We can write the **flow conservation** constraint as $\delta_x(i) = 0$. In general, we have flow **demand** constraints of the form

$$
\delta_x(i) = d_i, \quad i \in V,
$$

where the demand $d_i$ may be a constant (positive, negative, or zero) or a variable. Note that a negative demand is used to indicate a **supply**.

For example, in the [maximum flow problem](https://en.wikipedia.org/wiki/Maximum_flow_problem), the aim is to determine the maximum possible flow from a **source** node $s \in V$ to a **sink** node $t \in V$, subject to capacity constraints on the edges. We can write the maximum flow problem as

$$
\begin{aligned}
& \underset{x, f}{\text{maximize}} && f \\
&\text{subject to}&& \delta_x(s) = -f, \\
&&& \delta_x(t) = f, \\
&&& \delta_x(i) = 0, && i \in V\setminus\{s,t\}, \\
&&& 0 \leq x_{ij} \leq c_{ij}, && (i,j) \in E.
\end{aligned}
$$

Note that this is a linear program.

---
### Exercise 1

1. Install [GLPK](https://www.gnu.org/software/glpk/) (GNU Linear Programming Kit) on your computer. Make sure you can run the GLPK command `glpsol` at the command line:

                % glpsol
                GLPSOL--GLPK LP/MIP Solver 5.0
                No input problem file specified; try glpsol --help
                

2. Download the GMPL (GNU Mathematical Programming Language) model file `maxflow.mod` and the data file `maxflow.dat` from Blackboard. Run the  command `glpsol -m maxflow.mod -d maxflow.dat`. This finds the maximum flow from `KingswayBoundary` to `UBC` and writes the solution to a file called `sol.txt`. It will also generate a file called `nodes.txt` that we will use for making plots.

3. Using the information in the `nodes.txt` and `sol.txt` to create a plot of the road network and the optimal solution. Based on your plot, what is the bottleneck?

4. Solve the max flow problem again, but this time use the command `glpsol -m maxflow.mod -d maxflow.dat -o maxflow.sol`. In the generated file, `maxflow.sol`, the column labelled `Marginal` represents the dual multipliers for the constraints. The marginal value of each edge gives the rate of change on the optimal value obtained by increasing and/or decreasing the capacity of that edge. Which edges have the largest marginal values? How is this related to the bottleneck you identified?

---
### Exercise 2


1. Create `maxflow_multi.mod` and `maxflow_multi.dat` to find the maximum total flow from the following set of nine source nodes

              set SOURCES :=
              MarineGranville  MarineOak      MarineCambie
              KnightMarine     MarineBoundary KingswayBoundary
              LougheedBoundary Boundary1st    BoundaryHastings ;
              

   to the following set of two sink nodes

              set SINKS := Downtown UBC ;
              

   subject to the same capacity constraints as before. Solve the problem and create a plot of the flow.
   
2. Create a new data file `maxflow_skytrain.dat` which adds the Canada Line (CL) going between `MarineCambie` and `Downtown` through `Cambie41st` and `CambieBroadway` with a frequency of 12 trips per hour and a capacity of 200 passengers per trip, and the Expo/Millemium Line (EML) going between `KingswayBoundary` and `Downtown` through `KingswayKnight`, `KingswayBroadway`, and `MainTerminal`, with a frequency of 24 trips per hour and a capacity of 200 passengers per trip. Solve the problem and create a plot of the flow.

In [9]:
glpsol -m maxflow_multi.mod -d maxflow_multi.dat

LoadError: syntax: extra token "maxflow_multi" after end of expression

---
### Exercise 3

We will now find the **minimum cost flow** that satisfies certain supply/demand constraints on the nodes and capacity constraints on the edges, using the costs described below.  Start by creating copies of `maxflow.mod` and `maxflow_skytrain.dat`, and naming them `mincost.mod` and `mincost.dat`.

1. Suppose it costs \$0.10 to transport one passenger one kilometre. Add the following to your `mincost.dat` file:

              param unitcost := .10;  # cost to transport a passenger per km
              

   In your `mincost.mod` file, use the haversine formula to compute the length of each edge in kilometres from the longitude/latitude coordinates of its endpoints. 

              param pi := 3.14159265358979;
              param R := 6371;            # mean radius of the Earth in km

              param a{(i,j) in EDGES} := 
                     sin(pi*( (coord[i,'lat'] - coord[j,'lat'])/2 )/180)^2 + 
                     (
                     cos(pi*coord[i,'lat']/180) * 
                     cos(pi*coord[j,'lat']/180) * 
                     sin(pi*( (coord[i,'long'] - coord[j,'long'])/2 )/180)^2
                     ) ;

              param dist{(i,j) in EDGES} := 
                     2*R*atan( sqrt(a[i,j]), sqrt(1-a[i,j]) );
                     

   Use the length of each edge to compute the cost of transporting one passenger along that edge, where you should define cost in your `mincost.mod` file as:

              param cost{(i,j) in EDGES} := unitcost * dist[i,j];
              

   We want to minimize the total cost per hour:

              minimize total_cost: sum{(i,j) in EDGES} cost[i,j] * Flow[i,j];
        
2. Set supply/demand constraints as follows. Put 

              param demand {NODES} default 0;
              

   in your `mincost.mod` file and put

              param demand :=
               MarineGranville            -500
                     MarineOak            -500
                  MarineCambie            -500
              KingswayBoundary            -500
              BoundaryHastings            -500
                           UBC            2500 ;
                            
   in your `mincost.dat` file. All other parameters are as in your `maxflow_skytrain.dat` file. Solve the problem and create a plot of the flow.

3. The dual multiplier for each edge gives the rate of change of the minimum cost per unit increase in the capacity of the edge. After solving the problem in the model file, the value of the dual multiplier is stored in `Flow[i,j].dual`. Which edge has the most negative dual multiplier? Based on this, how would you recommend the transit network be improved to reduce the operating cost required to meet the above demands? Create a new data file, `mincost2.dat`, that reflects your proposed improvement and find the new operating cost.

---
### Exercise 4

We will now consider separate flows for travellers going to UBC and travellers going downtown. Such a problem is known as a **minimum cost multi-commodity flow** problem.

1. Start by creating copies of `mincost.mod` and `mincost.dat`, and naming them `multicommodity.mod` and `multicommodity.dat`. Given a set of destinations, we add a destination index to the `Flow` variable. Put the following into your `multicommodity.mod` file.

                set DESTINATIONS within NODES;
                var Flow{EDGES, DESTINATIONS} >=0;
                

2. We now have separate demands for each type of flow. Put the following into your `multicommodity.dat` file.

                set DESTINATIONS := Downtown UBC ;
                param demand :               Downtown            UBC :=
                    KingswayBoundary            -3000          -2000
                        MarineCambie            -1000           -500
                     MarineGranville             -500           -500
                           MarineOak             -500          -1000
                            Downtown             5000              0
                                 UBC                0           4000 ;
                                 

3. Update the objective function and demand constraints appropriately, and do not forget to include capacity constraints. Solve the problem and create a plot of the flow.

4. Analyze the solution using the dual multipliers and propose an improvement to the network.

---